# 03 — Modelos estadísticos

## Objetivo

Construir modelos estadísticos de referencia para estimar las principales
componentes del riesgo:

- frecuencia de siniestros;
- severidad;
- prima pura mediante frecuencia × severidad;
- costo agregado mediante un modelo directo.

Estos modelos funcionarán como *baseline* para comparar posteriormente métodos
de Machine Learning y Deep Learning.

La evaluación se realizará sobre una partición de prueba independiente y,
cuando sea posible, también contra las cantidades verdaderas generadas por el
DGP sintético.

Las variables *oracle* se utilizarán exclusivamente para evaluación y nunca
como predictores durante el entrenamiento.

### 1.1 Imports

In [31]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm
import statsmodels.formula.api as smf

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

SEED = 42

### 1.2 Carga de datos

In [32]:
DATA_PATH = Path("../data/raw/synthetic_insurance_portfolio.csv")

df = pd.read_csv(DATA_PATH)

print(f"Observaciones: {len(df):,}")
print(f"Variables: {df.shape[1]}")

Observaciones: 10,000
Variables: 15


In [33]:
# Etiquetas categóricas
df["zona_cat"] = df["zona"].map({
    1: "Muy bajo",
    2: "Bajo",
    3: "Medio",
    4: "Alto",
    5: "Muy alto"
})

df["cobertura_cat"] = df["cobertura"].map({
    0: "Limitada",
    1: "RC",
    2: "Amplia"
})

df["tipo_uso_cat"] = df["tipo_uso"].map({
    0: "Particular",
    1: "Transporte privado",
    2: "Carga comercial"
})

### 1.3 Definición de variables

In [34]:
# Variables independientes
FEATURES = [
    "edad",
    "zona",
    "anios_vehiculo",
    "cobertura",
    "historial_siniestros",
    "tipo_uso"
]

# Variables dependientes
TARGET_FREQUENCY = "numero_siniestros"
TARGET_SEVERITY = "severidad_promedio"
TARGET_COST = "costo_siniestros"

# Variables para validar resultados
ORACLE = [
    "lambda_real",
    "prob_siniestro_real",
    "severidad_esperada_real",
    "prima_pura_real"
]

# Validar que ninguna variable oracle puede entrar como predictor
assert not set(FEATURES) & set(ORACLE)

### 1.4 Train/Test split

In [35]:
train_idx, test_idx = train_test_split(
    df.index,
    test_size=0.20,
    random_state=SEED
)

df_train = df.loc[train_idx].copy()
df_test = df.loc[test_idx].copy()

print(f"Train: {len(df_train):,}")
print(f"Test:  {len(df_test):,}")

Train: 8,000
Test:  2,000


In [36]:
# Resumen del split, verificar que las frecuencias sean comparables
split_summary = pd.DataFrame({
    "train": [
        df_train["numero_siniestros"].mean(),
        df_train["tuvo_siniestro"].mean(),
        df_train["costo_siniestros"].mean()
    ],
    "test": [
        df_test["numero_siniestros"].mean(),
        df_test["tuvo_siniestro"].mean(),
        df_test["costo_siniestros"].mean()
    ]
}, index=[
    "frecuencia_media",
    "proporcion_con_siniestro",
    "costo_medio"
])

split_summary

,train,test
frecuencia_media,0.154875,0.157500
proporcion_con_siniestro,0.141375,0.146000
costo_medio,1936.977709,2008.681614


In [37]:
split_df = pd.DataFrame({
    "policy_id": df["policy_id"],
    "split": np.where(
        df.index.isin(train_idx),
        "train",
        "test"
    )
})

split_df.head()

,policy_id,split
0,1,test
1,2,train
2,3,train
3,4,test
4,5,train


In [38]:
SPLIT_PATH = Path("../data/processed/train_test_split.csv")

SPLIT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

split_df.to_csv(
    SPLIT_PATH,
    index=False
)

## 2. Modelo GLM Poisson para frecuencia

La variable `numero_siniestros` representa un conteo no negativo. El análisis
exploratorio mostró además que su media y varianza marginales son similares,
por lo que un modelo Poisson constituye un punto de partida natural.

Se supone que:

$$
N_i \mid X_i \sim Poisson(\lambda_i)
$$

y se utiliza un enlace logarítmico:

$$
\log(\lambda_i)=X_i^\top\beta.
$$

De esta manera:

$$
E[N_i\mid X_i]
=
\lambda_i
=
\exp(X_i^\top\beta).
$$

La exponenciación de los coeficientes permite interpretar los parámetros como
efectos multiplicativos sobre la frecuencia esperada.

In [39]:
# Formula que indica la regresión lineal poisson
formula_poisson_1 = """
numero_siniestros ~
edad +
anios_vehiculo +
C(zona_cat) +
C(cobertura_cat) +
historial_siniestros +
C(tipo_uso_cat)
"""

In [40]:
# Ajuste del GLM Poisson
poisson_1 = smf.glm(
    formula=formula_poisson_1,
    data=df_train,
    family=sm.families.Poisson()
).fit()

print(poisson_1.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:      numero_siniestros   No. Observations:                 8000
Model:                            GLM   Df Residuals:                     7988
Model Family:                 Poisson   Df Model:                           11
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -3471.6
Date:              lun., 21 sep. 2026   Deviance:                       4616.4
Time:                        21:51:54   Pearson chi2:                 7.86e+03
No. Iterations:                     6   Pseudo R-squ. (CS):            0.03822
Covariance Type:            nonrobust                                         
                                            coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------

### 2.3 Coeficientes como efectos multiplicativos

Una de las grandes ventajas del GLM Poisson con enlace log es:

$$
\log(\lambda) = X\beta
$$

por lo que: 

$$
\lambda = e^{X\beta}
$$

Entonces:

$$
e^{\beta_j}
$$

puede interpretarse como un **factor multiplicativo sobre la frecuencia esperada**, manteniendo lo demás constante.

In [41]:
coef_table = pd.DataFrame({
    "coef": poisson_1.params,
    "exp_coef": np.exp(poisson_1.params),
    "p_value": poisson_1.pvalues
})

coef_table

,coef,exp_coef,p_value
Intercept,-1.743861,0.174844,2.569805e-30
C(zona_cat)[T.Bajo],-0.254291,0.775466,4.602263e-03
C(zona_cat)[T.Medio],-0.226725,0.797140,1.207079e-02
C(zona_cat)[T.Muy alto],0.225852,1.253390,5.072698e-03
C(zona_cat)[T.Muy bajo],-0.449840,0.637730,2.276374e-06
C(cobertura_cat)[T.Limitada],-0.217514,0.804516,2.287038e-03
C(cobertura_cat)[T.RC],-0.188364,0.828313,1.636456e-02
C(tipo_uso_cat)[T.Particular],-0.244790,0.782869,1.343451e-02
C(tipo_uso_cat)[T.Transporte privado],-0.179523,0.835669,7.194929e-02
edad,-0.003776,0.996231,2.991666e-02


De acuerdo al primer ajuste que se realizo, podemos explicar lo siguiente:

- historial_siniestros (Coef: 0.3481 | $p < 0.001$): Por cada siniestro adicional en el historial del cliente, la frecuencia esperada de siniestros aumenta aproximadamente un $41.6\%$ ($\exp(0.3481) - 1$), manteniendose constantes las demás variables.

- anios_vehiculo (Coef: 0.0160 | $p = 0.001$): A mayor antigüedad del vehículo, mayor siniestralidad. Por cada año extra de antigüedad del auto, la frecuencia esperada aumenta un $1.6\%$ ($\exp(0.0160) - 1$).

- edad (Coef: -0.0038 | $p = 0.030$): Bajo la especificación lineal actual, el modelo estima una reducción promedio de aproximadamente $0.38\%$ ($\exp(-0.0038) - 1$) por año. Sin embargo, el EDA sugirió una relación no lineal, por lo que este coeficiente podría estar resumiendo inadecuadamente el efecto real de la edad.

 - zona_cat (Zona de riesgo): La categoría de referencia es "Alto". 
    - Muy alto: Presenta un aumento significativo del riesgo ($+25.3\%$) respecto a la zona de referencia.
    - Muy bajo, Bajo y Medio: Tienen un riesgo significativamente menor (por ejemplo, estar en una zona Muy bajo reduce el riesgo un $36.2\%$ respecto a la zona base).

- cobertura_cat (Tipo de cobertura): Manteniendo constantes las demás variables, las pólizas Limitadas presentan una frecuencia esperada aproximadamente 19.5% inferior a las de cobertura Amplia, mientras que RC presenta una frecuencia aproximadamente 17.2% inferior.

- tipo_uso_cat (Uso del vehículo)La categoría de referencia es "carga comercial".
   - El uso Particular reduce el riesgo un $21.7\%$ ($p = 0.013$).
   - El uso Transporte privado no alcanza significancia estadística del $5\%$ ($p = 0.072$), por lo que no hay suficiente evidencia para rechazar: $H_0: \beta_{\text{Transporte privado}}=0$

#### 2.3.1 Análisis de sobredispersión y Pseudo $R^2$

La dispersión de Pearson de de 0.984, aproximadamente 1 y menor que uno, eso no da una pista de que hay equidispersión, por lo que un modelo Poisson es una gran opción.

Respecto a la pseudo $R^2$ es de 0.038, pero esto tiene sentido ya que, la frecuencia media es de apenas 0.155, hay mucha variabilidad individual dificil de predecir.

Ejemplo:

Dos asegurados con exactamente el mismo $X$ pueden tener:

$$
N_1=0, N_2=1
$$

simplemente por azar.

In [42]:
pearson_dispersion = (
    poisson_1.pearson_chi2 /
    poisson_1.df_resid
)

print(f"Dispersión Pearson φ: {pearson_dispersion:.4f}")
print(f"Pseudo R^2: {0.03822}")

Dispersión Pearson φ: 0.9841
Pseudo R^2: 0.03822


### 2.4 Mejora del modelo: efecto no lineal de edad

En el Análisis explotatorio observamos que:

$$
\text{Frecuencia}\uparrow \text{hasta edades intermedia y después, Frecuencia}\downarrow
$$

Pero el modelo actual impone una recta:

$$
\log(\lambda) = \beta_0 + B_1edad + ...
$$

Pero el comportamiento de edad es similar a una parabola, por lo que haremos:

$$
\text{edad}^2.
$$

Pero primero vamos a restarle media de la edad del portafolio, para evitar problemas de multicolinealidad.


#### Demostración de restar media a una variable evita tener multicolinealidad

Sea $Z = X - \mu_X$, de modo que por construcción $E[Z] = 0$.
Queremos calcular la covarianza entre la variable centrada $Z$ y su cuadrado $Z^2$:

$$\text{Cov}(Z, Z^2) = E[(Z - E[Z])(Z^2 - E[Z^2])]$$

Como $E[Z] = 0$, esto se simplifica a:

$$\text{Cov}(Z, Z^2) = E[Z \cdot (Z^2 - E[Z^2])]$$

$$\text{Cov}(Z, Z^2) = E[Z^3] - E[Z] \cdot E[Z^2]$$

Como $E[Z] = 0$:
$$\text{Cov}(Z, Z^2) = E[Z^3]$$

$E[Z^3]$ es el tercer momento central (relacionado con el sesgo o skewness).Si la distribución de edad es simétrica alrededor de su media, $E[Z^3] = 0$.Por lo tanto, $\text{Cov}(Z, Z^2) = 0$, haciendo que la correlación sea exactamente 0.

In [ ]:
edad_media_train = df_train["edad"].mean()

df_train["edad_c"] = (
    df_train["edad"] - edad_media_train
)

df_test["edad_c"] = (
    df_test["edad"] - edad_media_train
)

df_train["edad_c2"] = df_train["edad_c"] ** 2
df_test["edad_c2"] = df_test["edad_c"] ** 2

In [44]:
formula_poisson_2 = """
numero_siniestros ~
edad_c +
edad_c2 +
anios_vehiculo +
C(zona_cat) +
C(cobertura_cat) +
historial_siniestros +
C(tipo_uso_cat)
"""

In [45]:
poisson_2 = smf.glm(
    formula=formula_poisson_2,
    data=df_train,
    family=sm.families.Poisson()
).fit()

print(poisson_2.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:      numero_siniestros   No. Observations:                 8000
Model:                            GLM   Df Residuals:                     7987
Model Family:                 Poisson   Df Model:                           12
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -3463.3
Date:              lun., 21 sep. 2026   Deviance:                       4599.8
Time:                        21:52:07   Pearson chi2:                 7.83e+03
No. Iterations:                     6   Pseudo R-squ. (CS):            0.04022
Covariance Type:            nonrobust                                         
                                            coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------

In [46]:
comparison_poisson = pd.DataFrame({
    "Modelo": [
        "Poisson lineal",
        "Poisson + edad²"
    ],
    "LogLik": [
        poisson_1.llf,
        poisson_2.llf
    ],
    "AIC": [
        poisson_1.aic,
        poisson_2.aic
    ],
    "Deviance": [
        poisson_1.deviance,
        poisson_2.deviance
    ],
    "Pearson dispersion": [
        poisson_1.pearson_chi2 / poisson_1.df_resid,
        poisson_2.pearson_chi2 / poisson_2.df_resid
    ]
})

comparison_poisson

,Modelo,LogLik,AIC,Deviance,Pearson dispersion
0,Poisson lineal,-3471.603440,6967.206881,4616.444300,0.984119
1,Poisson + edad²,-3463.288926,6952.577851,4599.815271,0.980739
